In [16]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser


import pickle, os
import pandas as pd
import numpy as np


In [17]:
event_log_name = "p2p"
log_path = f"D:\\LTNcoder\\.out\\eventlogs\\{event_log_name}-0.3-1.xes"

event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

parsing log, completed traces ::   0%|          | 0/5000 [00:00<?, ?it/s]

In [18]:
# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename=f'{event_log_name}_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename=f'{event_log_name}_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

In [ ]:
if not os.path.exists(f"{event_log_name}-0.3-1.decl"):
    print(f"File {event_log_name}-0.3-1.decl does not exist, running discovery...")
    discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
    declare_model: DeclareModel = discovery.run()
    print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
    model_constraints = declare_model.get_decl_model_constraints()
    declare_model.to_file(f"{event_log_name}-0.3-1.decl")

In [20]:
if not os.path.exists(f'{event_log_name}_5000_conformance_results.pkl') and event_log is not None and declare_model is not None:
    print(f"File {event_log_name}_5000_conformance_results.pkl does not exist, running conformance checking...")
    basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
    conf_check_res: MPDeclareResultsBrowser = basic_checker.run()
    save_conformance_results(conf_check_res)
else:
    print(f"Loading conformance checking results from {event_log_name}_5000_conformance_results.pkl")
    conf_check_res = load_conformance_results()
conf_check_df =  conf_check_res.get_metric(metric="state")


Loading conformance checking results from p2p_5000_conformance_results.pkl
Conformance checking results loaded from p2p_5000_conformance_results.pkl


In [21]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    # 'activation_rate': activation_rate
})
# metrics_df


C:\Users\devas\AppData\Local\Temp\ipykernel_79036\1810640346.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [22]:
# filtered_metrics_df is metrics_df but with rows with support less than 0.2 and in descending order of confidence and no "not" in the constraint name
filtered_metrics_df = metrics_df[~metrics_df.index.str.contains("Not") & (metrics_df['support'] <= 0.2) & (metrics_df['confidence'] >= 0.7)].sort_values(by='confidence', ascending=False)
# filtered_metrics_df = metrics_df[metrics_df['support'] <= 0.2].sort_values(by='confidence', ascending=False)
print("Filtered Metrics DataFrame:")
display(filtered_metrics_df)

Filtered Metrics DataFrame:


,support,confidence
"Responded Existence[Create PR, Pay] | |",0.1534,0.994812
"Response[Create PR, Pay] | |",0.1532,0.993515
"Responded Existence[Approve PO 2, Post IR] | |",0.0756,0.992126
"Response[Approve PO 2, Pay] | |",0.0756,0.992126
"Responded Existence[Approve PO 2, Pay] | |",0.0756,0.992126
...,...,...
"Alternate Precedence[Create SC, Approve PO 2] | |",0.0638,0.837270
"Precedence[Approve SC, Approve PO 2] | |",0.0638,0.837270
"Alternate Precedence[Approve SC, Approve PO 2] | |",0.0634,0.832021
"Precedence[Purchase SC, Approve PO 2] | |",0.0634,0.832021


# Constraints with low support and high confidence
1. Responded Existence[Create PR, Pay] | |	0.1534	0.9948119325551232	0.1542
2. Responded Existence[Approve PO 2, Post IR] | |	0.0756	0.9921259842519685	0.0762
3. Response[Approve PO 2, Pay] | |	0.0756	0.9921259842519685	0.0762
4. Response[Approve PO 2, Release PO] | |	0.073	0.958005249343832	0.0762 
5. *Response[Release PR, Pay] | |	0.1536	0.9858793324775353*

In [23]:
interesting_constraints = [
    # "Response[Approve PO 2, Release PO] | |",
    "Response[Release PR, Pay] | |"
    ]

In [24]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)
non_zero_rows = state_df.index[state_df[interesting_constraints[0]] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
with open(f'{event_log_name}_ltn_rows.pkl', 'wb') as f:
    pickle.dump(non_zero_rows, f)

Response[Release PR, Pay] | |    768
dtype: int64
[1, 6, 7, 8, 16, 25, 28, 35, 37, 38, 67, 70, 73, 92, 94, 97, 99, 101, 106, 109, 118, 120, 128, 132, 156, 157, 175, 178, 179, 182, 186, 189, 201, 204, 205, 210, 213, 218, 225, 227, 237, 247, 254, 261, 264, 267, 271, 272, 281, 288, 290, 292, 296, 306, 307, 327, 334, 336, 338, 348, 351, 354, 359, 363, 367, 385, 388, 396, 402, 403, 411, 422, 437, 444, 446, 449, 452, 456, 463, 485, 489, 492, 495, 499, 516, 533, 534, 544, 550, 554, 565, 567, 568, 577, 578, 579, 610, 616, 623, 632, 635, 646, 656, 659, 660, 669, 688, 699, 705, 710, 713, 719, 737, 740, 752, 754, 762, 764, 765, 770, 779, 786, 795, 802, 806, 808, 825, 835, 842, 843, 849, 850, 858, 859, 876, 878, 880, 882, 883, 885, 887, 890, 922, 964, 971, 975, 978, 988, 994, 1003, 1004, 1005, 1008, 1010, 1013, 1036, 1052, 1060, 1071, 1074, 1084, 1088, 1099, 1106, 1114, 1116, 1127, 1128, 1151, 1156, 1161, 1165, 1168, 1169, 1184, 1194, 1199, 1203, 1211, 1212, 1229, 1245, 1248, 1261, 1265, 1274, 128

In [25]:
print("END")

END


In [26]:
# summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
# summary_df = summary_df.reindex([0, 1])
# summary_df = summary_df / len(conf_check_df)
# summary_df = summary_df.T
# summary_df = summary_df.sort_values(by=1, ascending=False)
# display(summary_df)